In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_wiki_un_members():
    url = "https://en.wikipedia.org/wiki/Member_states_of_the_United_Nations"
    base_url = "https://en.wikipedia.org"
    
    # Add a standard user-agent to ensure the server processes the request properly
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    # 1. Fetch the webpage content
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to retrieve the page. Status code: {response.status_code}")
        return
        
    # 2. Parse the HTML using Beautiful Soup
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 3. Target the data tables
    tables = soup.find_all('table', class_='wikitable')
    
    country_data = []
    
    for table in tables:
        rows = table.find_all('tr')
        
        if not rows:
            continue
            
        # Verify if this is the target table by checking the header row text
        header = rows[0].text.lower()
        if "member state" in header and "date of admission" in header:
            
            # Iterate through the rows, skipping the initial header
            for row in rows[1:]:
                # The country name is located in the first <th> or <td> cell of the row
                first_cell = row.find(['th', 'td'])
                
                if first_cell:
                    # Find all links within the cell
                    links = first_cell.find_all('a')
                    
                    for link_tag in links:
                        country_name = link_tag.text.strip()
                        
                        # The flag icon link has no text. If text exists, it is the country link.
                        if country_name:
                            # Wikipedia hrefs are relative (e.g., '/wiki/Afghanistan'), 
                            # so we append them to the base URL
                            country_url = base_url + link_tag['href']
                            
                            country_data.append({
                                'name': country_name, 
                                'url': country_url
                            })
                            break # Move to the next row once the correct link is processed
            
            # Break the outer loop once the main table has been fully parsed
            break

    # 4. Output the extracted data
    print(f"Total countries found: {len(country_data)}\n")
    for i, data in enumerate(country_data, 1):
         print(f"{i}. {data['name']} - {data['url']}")

    countries = pd.DataFrame(country_data)
    countries.to_csv('country_info.csv', index=False)


if __name__ == "__main__":
    scrape_wiki_un_members()

Total Member States found: 193

1. Afghanistan - https://en.wikipedia.org/wiki/Afghanistan
2. Albania - https://en.wikipedia.org/wiki/Albania
3. Algeria - https://en.wikipedia.org/wiki/Algeria
4. Andorra - https://en.wikipedia.org/wiki/Andorra
5. Angola - https://en.wikipedia.org/wiki/Angola
6. Antigua and Barbuda - https://en.wikipedia.org/wiki/Antigua_and_Barbuda
7. Argentina - https://en.wikipedia.org/wiki/Argentina
8. Armenia - https://en.wikipedia.org/wiki/Armenia
9. Australia - https://en.wikipedia.org/wiki/Australia
10. Austria - https://en.wikipedia.org/wiki/Austria
11. Azerbaijan - https://en.wikipedia.org/wiki/Azerbaijan
12. Bahamas, The - https://en.wikipedia.org/wiki/The_Bahamas
13. Bahrain - https://en.wikipedia.org/wiki/Bahrain
14. Bangladesh - https://en.wikipedia.org/wiki/Bangladesh
15. Barbados - https://en.wikipedia.org/wiki/Barbados
16. Belarus - https://en.wikipedia.org/wiki/Belarus
17. Belgium - https://en.wikipedia.org/wiki/Belgium
18. Belize - https://en.wikipedi